In [0]:

# ============================================================
# CELDA 1. PRUEBA DE CONEXIÓN CON OPEN-METEO
# ============================================================

# Comprobamos que Databricks pueda consultar la API
# de Open-Meteo antes de comenzar la ingesta completa.
#
# Fuente: Open-Meteo Previous Runs API
# Modelo meteorológico: GFS Global
# Resolución: 1 hora
# Pronóstico: 24 horas antes de la hora pronosticada

import requests
import pandas as pd

# Dirección de la API de pronósticos históricos.
URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

# Coordenadas del aeropuerto JFK, Nueva York.
LATITUD = 40.6413
LONGITUD = -73.7781

# Variables meteorológicas seleccionadas después
# de comprobar su cobertura en Google Colab.
VARIABLES = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

# Solicitamos los pronósticos disponibles 24 horas
# antes de la hora meteorológica correspondiente.
VARIABLES_API = [
    f"{variable}_previous_day1"
    for variable in VARIABLES
]

# Probamos una semana de febrero de 2024.
parametros = {
    "latitude": LATITUD,
    "longitude": LONGITUD,
    "start_date": "2024-02-01",
    "end_date": "2024-02-07",
    "hourly": ",".join(VARIABLES_API),
    "timezone": "UTC",
    "models": "gfs_global"
}

# Consultamos la API desde Databricks.
respuesta = requests.get(
    URL,
    params=parametros,
    timeout=90
)

respuesta.raise_for_status()

datos = respuesta.json()

# Comprobamos que la respuesta incluya registros horarios.
if datos.get("error"):
    raise ValueError(datos.get("reason", "Error de API"))

horarios = datos.get("hourly", {})

df_prueba = pd.DataFrame(horarios)

# Agregamos el aeropuerto para identificar los datos.
df_prueba["airport"] = "JFK"

print("Conexión exitosa con Open-Meteo.")
print("Aeropuerto:", "JFK")
print("Registros horarios:", len(df_prueba))
print("Columnas:", len(df_prueba.columns))

# Mostramos los primeros registros para revisar
# que las variables tengan valores disponibles.
display(df_prueba.head(10))


In [0]:

# ============================================================
# CELDA 2. IDENTIFICAR AEROPUERTOS DE ORIGEN EN BTS
# ============================================================

# Consultamos la tabla Bronze de vuelos que ya cargó
# el equipo para identificar los aeropuertos que
# necesitaremos relacionar con datos meteorológicos.

from pyspark.sql import functions as F

# Tabla original de vuelos del proyecto.
TABLA_BTS = "workspace.bronze_flights.bts_flights_raw"

# Como el modelo con clima utilizará 2024 y 2025,
# analizamos únicamente los vuelos de esos años.
vuelos = (
    spark.table(TABLA_BTS)
    .filter(F.col("_source_year").isin(2024, 2025))
)

# Contamos los vuelos por aeropuerto de origen.
aeropuertos = (
    vuelos
    .filter(F.col("ORIGIN").isNotNull())
    .filter(F.trim(F.col("ORIGIN")) != "")
    .groupBy("ORIGIN")
    .agg(F.count("*").alias("total_vuelos"))
    .orderBy(F.desc("total_vuelos"))
)

# Mostramos cuántos aeropuertos distintos hay.
total_aeropuertos = aeropuertos.count()

print("Aeropuertos de origen distintos:", total_aeropuertos)

# Mostramos los 20 aeropuertos con más vuelos.
print("\nTOP 20 AEROPUERTOS POR VOLUMEN DE VUELOS")

display(aeropuertos.limit(20))

# Calculamos el volumen total representado.
total_vuelos = (
    aeropuertos
    .agg(F.sum("total_vuelos").alias("total"))
    .first()["total"]
)

print("\nVuelos analizados:", f"{total_vuelos:,}")


In [0]:

# ============================================================
# CELDA 3. IDENTIFICAR COORDENADAS DE AEROPUERTOS
# ============================================================

# Instalamos un catálogo de aeropuertos con códigos
# IATA, latitud, longitud y ubicación geográfica.

%pip install airportsdata


In [0]:
dbutils.library.restartPython()

In [0]:

# ============================================================
# CELDA 6. COORDENADAS DE LOS AEROPUERTOS BTS
# ============================================================

# Relacionamos los códigos de aeropuerto que aparecen
# en BTS con sus coordenadas geográficas.
#
# Utilizamos airportsdata para obtener latitud, longitud
# y zona horaria sin introducirlos manualmente.

import airportsdata
import pandas as pd

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. RECUPERAR LOS AEROPUERTOS BTS
# ------------------------------------------------------------

# Volvemos a consultar Bronze porque reiniciamos Python.
TABLA_BTS = "workspace.bronze_flights.bts_flights_raw"

aeropuertos_bts = (
    spark.table(TABLA_BTS)
    .filter(F.col("_source_year").isin(2024, 2025))
    .filter(F.col("ORIGIN").isNotNull())
    .filter(F.trim(F.col("ORIGIN")) != "")
    .select("ORIGIN")
    .distinct()
)

# Guardamos los códigos IATA como una lista.
codigos_bts = [
    fila["ORIGIN"]
    for fila in aeropuertos_bts.collect()
]

# ------------------------------------------------------------
# 2. CONSULTAR EL CATÁLOGO DE AEROPUERTOS
# ------------------------------------------------------------

# El catálogo utiliza códigos IATA como JFK, ATL o DEN.
catalogo = airportsdata.load("IATA")

resultados = []

for codigo in codigos_bts:

    aeropuerto = catalogo.get(codigo)

    # Verificamos que exista y que esté ubicado en EE. UU.
    if aeropuerto and aeropuerto.get("country") == "US":

        resultados.append({
            "airport": codigo,
            "latitude": aeropuerto["lat"],
            "longitude": aeropuerto["lon"],
            "timezone": aeropuerto["tz"],
            "encontrado": True
        })

    else:

        resultados.append({
            "airport": codigo,
            "latitude": None,
            "longitude": None,
            "timezone": None,
            "encontrado": False
        })

# ------------------------------------------------------------
# 3. REVISAR LOS RESULTADOS
# ------------------------------------------------------------

df_coordenadas = pd.DataFrame(resultados)

encontrados = int(df_coordenadas["encontrado"].sum())
faltantes = df_coordenadas[
    ~df_coordenadas["encontrado"]
].copy()

print("=" * 50)
print("COBERTURA DE COORDENADAS")
print("=" * 50)

print("Aeropuertos BTS:", len(df_coordenadas))
print("Aeropuertos con coordenadas:", encontrados)
print("Aeropuertos sin coordenadas:", len(faltantes))

print("\nAEROPUERTOS SIN COORDENADAS")
display(faltantes)

print("\nEJEMPLOS DE AEROPUERTOS ENCONTRADOS")
display(
    df_coordenadas[
        df_coordenadas["encontrado"]
    ].head(10)
)


In [0]:

# ============================================================
# CELDA 7. COMPLETAR AEROPUERTOS DE TERRITORIOS DE EE. UU.
# ============================================================

# Agregamos las coordenadas de los tres aeropuertos
# que no pasaron nuestro filtro inicial.
#
# Las coordenadas provienen de registros aeroportuarios
# publicados y están expresadas en grados decimales.

coordenadas_adicionales = {
    "GUM": {
        "latitude": 13.4839519,
        "longitude": 144.7971294,
        "timezone": "Pacific/Guam"
    },
    "PPG": {
        "latitude": -14.3316622,
        "longitude": -170.7115031,
        "timezone": "Pacific/Pago_Pago"
    },
    "SPN": {
        "latitude": 15.1202486,
        "longitude": 145.7299864,
        "timezone": "Pacific/Saipan"
    }
}

# Completamos únicamente los aeropuertos faltantes.
# Conservamos las coordenadas que ya encontramos.

for codigo, datos in coordenadas_adicionales.items():

    mascara = df_coordenadas["airport"] == codigo

    for columna, valor in datos.items():
        df_coordenadas.loc[mascara, columna] = valor

    df_coordenadas.loc[mascara, "encontrado"] = True

# Validamos que no queden aeropuertos sin coordenadas.
total = len(df_coordenadas)

encontrados = int(
    df_coordenadas["encontrado"].sum()
)

faltantes = total - encontrados

print("RESULTADO FINAL")
print("-" * 40)
print("Aeropuertos BTS:", total)
print("Con coordenadas:", encontrados)
print("Sin coordenadas:", faltantes)

display(
    df_coordenadas[
        df_coordenadas["airport"].isin(
            ["GUM", "PPG", "SPN"]
        )
    ]
)

if encontrados == 358 and faltantes == 0:
    print("VALIDACIÓN EXITOSA: 358 aeropuertos identificados.")
else:
    print("REVISAR: todavía faltan coordenadas.")


In [0]:

# ============================================================
# CELDA 8. DIMENSIONAMIENTO DE LA INGESTA METEOROLÓGICA
# ============================================================

# Calculamos el volumen aproximado de datos que
# necesitaremos descargar antes de ejecutar la ingesta.
#
# No consultamos Open-Meteo en esta celda.
# Solamente estimamos el tamaño del procesamiento.

import pandas as pd

# Periodo definido para el modelo con clima.
FECHA_INICIO = "2024-02-01"
FECHA_FIN = "2025-12-31"

# Contamos los aeropuertos que ya identificamos.
total_aeropuertos = len(df_coordenadas)

# Calculamos todas las horas del periodo.
horas = pd.date_range(
    start=FECHA_INICIO,
    end=f"{FECHA_FIN} 23:00:00",
    freq="h",
    tz="UTC"
)

total_horas = len(horas)

# Número de meses que consultaremos.
periodos = pd.period_range(
    start="2024-02",
    end="2025-12",
    freq="M"
)

total_meses = len(periodos)

# Una consulta por aeropuerto y mes.
consultas_estimadas = (
    total_aeropuertos * total_meses
)

# Un registro por aeropuerto y hora.
registros_estimados = (
    total_aeropuertos * total_horas
)

# Mostramos el resumen del proyecto.
print("=" * 55)
print("DIMENSIONAMIENTO DEL BRONZE METEOROLÓGICO")
print("=" * 55)

print("Aeropuertos:", total_aeropuertos)
print("Meses:", total_meses)
print("Horas por aeropuerto:", f"{total_horas:,}")

print(
    "Consultas estimadas:",
    f"{consultas_estimadas:,}"
)

print(
    "Registros horarios estimados:",
    f"{registros_estimados:,}"
)

print("Variables meteorológicas:", 6)

print("\nEsta celda no descarga ni guarda datos.")


In [0]:

# ============================================================
# CELDA 9. PRUEBA DE CONSULTA MULTIAEROPUERTO
# ============================================================

# En lugar de realizar una solicitud por aeropuerto,
# probamos si Open-Meteo devuelve tres aeropuertos
# dentro de una sola respuesta.
#
# Todavía no guardamos nada en Bronze.

import requests
import pandas as pd

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

# Recuperamos las coordenadas que ya identificamos.
aeropuertos_prueba = (
    df_coordenadas[
        df_coordenadas["airport"].isin(["JFK", "DEN", "ATL"])
    ]
    .sort_values("airport")
    .reset_index(drop=True)
)

VARIABLES = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

parametros = {
    "latitude": ",".join(
        aeropuertos_prueba["latitude"].astype(str)
    ),
    "longitude": ",".join(
        aeropuertos_prueba["longitude"].astype(str)
    ),
    "start_date": "2025-07-01",
    "end_date": "2025-07-07",
    "hourly": ",".join(
        f"{v}_previous_day1" for v in VARIABLES
    ),
    "timezone": "UTC",
    "models": "gfs_global"
}

respuesta = requests.get(
    URL,
    params=parametros,
    timeout=120
)

respuesta.raise_for_status()
datos = respuesta.json()

# Al solicitar varias ubicaciones, esperamos una lista.
if not isinstance(datos, list):
    raise ValueError(
        "La API no devolvió una lista de ubicaciones."
    )

if len(datos) != len(aeropuertos_prueba):
    raise ValueError(
        "El número de ubicaciones no coincide."
    )

resultados = []

# Asociamos cada respuesta al aeropuerto correspondiente,
# respetando el orden de las coordenadas enviadas.
for indice, ubicacion in enumerate(datos):

    codigo = aeropuertos_prueba.loc[indice, "airport"]
    horarios = ubicacion.get("hourly", {})

    horas = len(horarios.get("time", []))

    variables_completas = sum(
        len(horarios.get(f"{v}_previous_day1", [])) == horas
        and all(
            valor is not None
            for valor in horarios.get(
                f"{v}_previous_day1", []
            )
        )
        for v in VARIABLES
    )

    resultados.append({
        "Aeropuerto": codigo,
        "Registros horarios": horas,
        "Variables completas": variables_completas
    })

print("RESULTADO DE CONSULTA MULTIAEROPUERTO")
display(pd.DataFrame(resultados))

if (
    len(resultados) == 3
    and all(
        r["Registros horarios"] == 168
        and r["Variables completas"] == 6
        for r in resultados
    )
):
    print("PRUEBA EXITOSA: tres aeropuertos en una solicitud.")
else:
    print("REVISAR: alguna ubicación tiene datos incompletos.")


In [0]:

# ============================================================
# CELDA 10. PRUEBA DE INGESTA POR LOTES
# ============================================================

# Probamos descargar un mes completo de pronósticos
# para 10 aeropuertos en una sola solicitud.
#
# Todavía no escribimos datos en Delta Lake.
# Esta prueba nos ayudará a definir el tamaño de los lotes.

import requests
import pandas as pd
import time

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

# Seleccionamos 10 aeropuertos con coordenadas disponibles.
lote = (
    df_coordenadas[
        df_coordenadas["encontrado"]
    ]
    .sort_values("airport")
    .head(10)
    .reset_index(drop=True)
)

variables = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

parametros = {
    "latitude": ",".join(lote["latitude"].astype(str)),
    "longitude": ",".join(lote["longitude"].astype(str)),
    "start_date": "2025-07-01",
    "end_date": "2025-07-31",
    "hourly": ",".join(
        f"{v}_previous_day1" for v in variables
    ),
    "timezone": "UTC",
    "models": "gfs_global"
}

# Medimos cuánto tarda la consulta.
inicio = time.time()

respuesta = requests.get(
    URL,
    params=parametros,
    timeout=180
)

respuesta.raise_for_status()

datos = respuesta.json()

duracion = round(time.time() - inicio, 2)

# Comprobamos que recibimos una respuesta por aeropuerto.
if not isinstance(datos, list) or len(datos) != len(lote):
    raise ValueError(
        "La cantidad de aeropuertos recibidos no coincide."
    )

resultados = []

for indice, ubicacion in enumerate(datos):

    aeropuerto = lote.loc[indice, "airport"]
    horarios = ubicacion.get("hourly", {})

    # Julio tiene 31 días: esperamos 744 horas.
    horas = len(horarios.get("time", []))

    variables_validas = sum(
        len(horarios.get(f"{v}_previous_day1", [])) == horas
        and all(
            valor is not None
            for valor in horarios.get(
                f"{v}_previous_day1", []
            )
        )
        for v in variables
    )

    resultados.append({
        "Aeropuerto": aeropuerto,
        "Horas recibidas": horas,
        "Variables completas": variables_validas
    })

tabla_lote = pd.DataFrame(resultados)

print("RESULTADO DE PRUEBA MENSUAL")
display(tabla_lote)

print("\nAeropuertos:", len(tabla_lote))
print("Registros horarios esperados:", 10 * 744)
print("Registros horarios recibidos:", tabla_lote["Horas recibidas"].sum())
print("Tiempo de consulta:", duracion, "segundos")

if (
    len(tabla_lote) == 10
    and (tabla_lote["Horas recibidas"] == 744).all()
    and (tabla_lote["Variables completas"] == 6).all()
):
    print("PRUEBA EXITOSA: lote mensual completo.")
else:
    print("REVISAR: se encontraron datos incompletos.")


In [0]:

# ============================================================
# CELDA 11. CREAR TABLA BRONZE DE PRONÓSTICOS
# ============================================================

# Creamos la tabla Delta donde almacenaremos los
# pronósticos históricos obtenidos de Open-Meteo.
#
# Cada registro representará un aeropuerto y una hora.
# Conservamos las seis variables de la fuente, así como
# metadatos para identificar el origen de cada registro.
#
# IF NOT EXISTS evita reemplazar una tabla existente.
# Todavía no insertamos ningún dato.

TABLA_CLIMA = "workspace.bronze_flights.weather_forecasts_raw"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA_CLIMA} (

    airport STRING,

    latitude DOUBLE,
    longitude DOUBLE,

    forecast_valid_time_utc STRING,

    temperature_2m_previous_day1 DOUBLE,
    precipitation_previous_day1 DOUBLE,
    wind_speed_10m_previous_day1 DOUBLE,
    relative_humidity_2m_previous_day1 DOUBLE,
    cloud_cover_previous_day1 DOUBLE,
    pressure_msl_previous_day1 DOUBLE,

    source_model STRING,
    source_api STRING,
    forecast_lead_hours INT,

    source_year INT,
    source_month INT,

    ingestion_timestamp TIMESTAMP

)
USING DELTA
""")

# Comprobamos que la tabla exista en Unity Catalog.
spark.sql(f"DESCRIBE TABLE {TABLA_CLIMA}").show(
    truncate=False
)

print("Tabla Bronze meteorológica preparada.")
print("Destino:", TABLA_CLIMA)


In [0]:

# ============================================================
# CELDA 12. PRIMERA CARGA DE PRONÓSTICOS A BRONZE
# ============================================================

# Descargamos julio de 2025 para 10 aeropuertos.
# Guardamos los pronósticos en nuestra tabla Delta.
#
# Utilizamos MERGE para evitar insertar dos veces
# el mismo aeropuerto y la misma hora si repetimos
# esta prueba.

import requests
import pandas as pd

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. CONFIGURAR LA CONSULTA
# ------------------------------------------------------------

TABLA_CLIMA = "workspace.bronze_flights.weather_forecasts_raw"

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

VARIABLES = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

# Recuperamos los 10 aeropuertos con coordenadas.
lote = (
    df_coordenadas[
        df_coordenadas["encontrado"]
    ]
    .sort_values("airport")
    .head(10)
    .reset_index(drop=True)
)

parametros = {
    "latitude": ",".join(
        lote["latitude"].astype(str)
    ),
    "longitude": ",".join(
        lote["longitude"].astype(str)
    ),
    "start_date": "2025-07-01",
    "end_date": "2025-07-31",
    "hourly": ",".join(
        f"{v}_previous_day1" for v in VARIABLES
    ),
    "timezone": "UTC",
    "models": "gfs_global"
}

# ------------------------------------------------------------
# 2. DESCARGAR EL LOTE
# ------------------------------------------------------------

respuesta = requests.get(
    URL,
    params=parametros,
    timeout=180
)

respuesta.raise_for_status()
datos = respuesta.json()

if not isinstance(datos, list) or len(datos) != len(lote):
    raise ValueError(
        "La API no devolvió los 10 aeropuertos esperados."
    )

# ------------------------------------------------------------
# 3. PREPARAR LOS REGISTROS
# ------------------------------------------------------------

registros = []

for indice, ubicacion in enumerate(datos):

    aeropuerto = lote.loc[indice, "airport"]
    latitude = float(lote.loc[indice, "latitude"])
    longitude = float(lote.loc[indice, "longitude"])

    horarios = ubicacion.get("hourly", {})
    fechas = horarios.get("time", [])

    if len(fechas) != 744:
        raise ValueError(
            f"{aeropuerto}: se esperaban 744 horas."
        )

    # Comprobamos que las seis variables tengan
    # una observación por cada hora del mes.
    for variable in VARIABLES:

        columna = f"{variable}_previous_day1"
        valores = horarios.get(columna, [])

        if len(valores) != 744:
            raise ValueError(
                f"{aeropuerto}: datos incompletos "
                f"para {columna}."
            )

    # Creamos una fila por aeropuerto y hora.
    for posicion, fecha in enumerate(fechas):

        fila = {
            "airport": aeropuerto,
            "latitude": latitude,
            "longitude": longitude,
            "forecast_valid_time_utc": fecha,

            "source_model": "gfs_global",
            "source_api": "open_meteo_previous_runs",
            "forecast_lead_hours": 24,

            "source_year": 2025,
            "source_month": 7
        }

        for variable in VARIABLES:

            columna = f"{variable}_previous_day1"
            valor = horarios[columna][posicion]

            fila[columna] = (
                float(valor)
                if valor is not None
                else None
            )

        registros.append(fila)

# ------------------------------------------------------------
# 4. CONVERTIR A SPARK
# ------------------------------------------------------------

df_pandas = pd.DataFrame(registros)

if len(df_pandas) != 7440:
    raise ValueError(
        "La cantidad de registros no coincide."
    )

# Validamos que no existan combinaciones repetidas
# de aeropuerto y hora en el lote descargado.
duplicados_lote = df_pandas.duplicated(
    subset=["airport", "forecast_valid_time_utc"]
).sum()

if duplicados_lote > 0:
    raise ValueError(
        f"Se encontraron {duplicados_lote} registros "
        "duplicados en la descarga."
    )

df_spark = (
    spark.createDataFrame(df_pandas)
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

# Ajustamos las columnas y los tipos a nuestra
# tabla Bronze ya creada.
columnas_destino = spark.table(TABLA_CLIMA).schema

df_spark = df_spark.select([
    F.col(campo.name).cast(campo.dataType).alias(campo.name)
    for campo in columnas_destino.fields
])

# ------------------------------------------------------------
# 5. INSERTAR SIN DUPLICAR REGISTROS
# ------------------------------------------------------------

# MERGE permite insertar registros nuevos y conservar
# los que ya estén almacenados en la tabla.
#
# La llave considera aeropuerto, hora, modelo
# y horizonte del pronóstico.

df_spark.createOrReplaceTempView(
    "weather_batch_temp"
)

columnas = df_spark.columns

columnas_sql = ", ".join(
    f"`{columna}`" for columna in columnas
)

valores_sql = ", ".join(
    f"s.`{columna}`" for columna in columnas
)

spark.sql(f"""
MERGE INTO {TABLA_CLIMA} AS t
USING weather_batch_temp AS s

ON t.airport = s.airport
AND t.forecast_valid_time_utc = s.forecast_valid_time_utc
AND t.source_model = s.source_model
AND t.forecast_lead_hours = s.forecast_lead_hours

WHEN NOT MATCHED THEN
INSERT ({columnas_sql})
VALUES ({valores_sql})
""")

# ------------------------------------------------------------
# 6. VALIDAR LO QUE QUEDÓ GUARDADO
# ------------------------------------------------------------

df_guardado = (
    spark.table(TABLA_CLIMA)
    .filter(F.col("source_year") == 2025)
    .filter(F.col("source_month") == 7)
)

print("=" * 50)
print("RESULTADO DEL PRIMER LOTE BRONZE")
print("=" * 50)

print("Registros descargados:", len(df_pandas))
print("Registros en tabla, julio 2025:", df_guardado.count())

print(
    "Aeropuertos almacenados:",
    df_guardado.select("airport").distinct().count()
)

display(
    df_guardado
    .groupBy("airport")
    .count()
    .orderBy("airport")
)

print("Primera carga de prueba finalizada.")


In [0]:

# ============================================================
# CELDA 13. GUARDAR COORDENADAS DE AEROPUERTOS EN BRONZE
# ============================================================

# Guardamos las coordenadas de los 358 aeropuertos BTS
# para utilizarlas durante la descarga de Open-Meteo.
#
# Aunque después seleccionemos solamente 30 aeropuertos
# para el modelo, conservamos el catálogo completo.
#
# Este código no modifica los datos de vuelos ni de clima.

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

TABLA_AEROPUERTOS = (
    "workspace.bronze_flights.airport_coordinates"
)

# ------------------------------------------------------------
# 1. PREPARAR EL CATÁLOGO
# ------------------------------------------------------------

# Recuperamos las coordenadas identificadas anteriormente.
df_catalogo = df_coordenadas[
    ["airport", "latitude", "longitude", "timezone"]
].copy()

# Convertimos los tipos para evitar problemas con Spark.
df_catalogo["airport"] = df_catalogo["airport"].astype(str)
df_catalogo["timezone"] = df_catalogo["timezone"].astype(str)

df_catalogo["latitude"] = pd.to_numeric(
    df_catalogo["latitude"], errors="coerce"
)

df_catalogo["longitude"] = pd.to_numeric(
    df_catalogo["longitude"], errors="coerce"
)

# ------------------------------------------------------------
# 2. VALIDAR LOS 358 AEROPUERTOS
# ------------------------------------------------------------

if len(df_catalogo) != 358:
    raise ValueError(
        f"Se esperaban 358 aeropuertos y hay {len(df_catalogo)}."
    )

if df_catalogo[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Hay aeropuertos sin coordenadas.")

if df_catalogo["airport"].duplicated().any():
    raise ValueError("Se encontraron aeropuertos duplicados.")

print("Validación exitosa: 358 aeropuertos preparados.")

# ------------------------------------------------------------
# 3. CONVERTIR LOS DATOS A SPARK
# ------------------------------------------------------------

# Definimos el esquema para conservar tipos consistentes.
esquema = StructType([
    StructField("airport", StringType(), False),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("timezone", StringType(), True)
])

# Convertimos las filas a tipos nativos de Python.
registros = [
    (
        str(fila.airport),
        float(fila.latitude),
        float(fila.longitude),
        str(fila.timezone)
    )
    for fila in df_catalogo.itertuples(index=False)
]

df_spark_aeropuertos = spark.createDataFrame(
    registros,
    schema=esquema
)

# ------------------------------------------------------------
# 4. GUARDAR EN DELTA LAKE
# ------------------------------------------------------------

# Solo creamos la tabla cuando todavía no existe.
# Si ya existe, NO insertamos otra vez los aeropuertos.

if spark.catalog.tableExists(TABLA_AEROPUERTOS):

    print("La tabla ya existe. No se duplicaron registros.")

else:

    (
        df_spark_aeropuertos.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_AEROPUERTOS)
    )

    print("Tabla de aeropuertos creada correctamente.")

# ------------------------------------------------------------
# 5. VALIDAR LA TABLA GUARDADA
# ------------------------------------------------------------

df_guardado = spark.table(TABLA_AEROPUERTOS)

total_guardado = df_guardado.count()

aeropuertos_unicos = (
    df_guardado
    .select("airport")
    .distinct()
    .count()
)

print("\nRESULTADO FINAL")
print("-" * 45)
print("Aeropuertos guardados:", total_guardado)
print("Aeropuertos únicos:", aeropuertos_unicos)
print("Tabla:", TABLA_AEROPUERTOS)

if total_guardado == 358 and aeropuertos_unicos == 358:
    print("VALIDACIÓN EXITOSA: catálogo completo.")
else:
    print("REVISAR: la tabla no tiene 358 aeropuertos únicos.")

display(
    df_guardado
    .orderBy("airport")
    .limit(10)
)


In [0]:

# ============================================================
# CELDA 14. SELECCIÓN DE LOS 30 AEROPUERTOS PRINCIPALES
# ============================================================

# Seleccionamos los aeropuertos según su número de vuelos,
# no según sus retrasos, para evitar sesgar la selección.
#
# Utilizamos el mismo periodo definido para el modelo
# meteorológico: febrero de 2024 a diciembre de 2025.

from pyspark.sql import functions as F

TABLA_BTS = "workspace.bronze_flights.bts_flights_raw"
TABLA_AEROPUERTOS = "workspace.bronze_flights.airport_coordinates"

# Recuperamos los vuelos del periodo seleccionado.
vuelos = (
    spark.table(TABLA_BTS)
    .filter(
        (F.col("_source_year") == 2025) |
        (
            (F.col("_source_year") == 2024) &
            (F.col("_source_month") >= 2)
        )
    )
)

# Contamos vuelos por aeropuerto de origen.
volumen = (
    vuelos
    .filter(F.col("ORIGIN").isNotNull())
    .groupBy("ORIGIN")
    .agg(F.count("*").alias("total_vuelos"))
    .orderBy(F.desc("total_vuelos"), F.asc("ORIGIN"))
)

# Seleccionamos los 30 con mayor número de vuelos.
top30 = volumen.limit(30)

# Relacionamos los aeropuertos con sus coordenadas.
coordenadas = spark.table(TABLA_AEROPUERTOS)

top30_coordenadas = (
    top30
    .join(
        coordenadas,
        top30["ORIGIN"] == coordenadas["airport"],
        "left"
    )
    .select(
        F.col("ORIGIN").alias("airport"),
        "total_vuelos",
        "latitude",
        "longitude",
        "timezone"
    )
    .orderBy(F.desc("total_vuelos"))
)

# Calculamos qué porcentaje de vuelos está representado.
total_vuelos = vuelos.count()

vuelos_top30 = (
    top30
    .agg(F.sum("total_vuelos").alias("total"))
    .first()["total"]
)

cobertura = vuelos_top30 / total_vuelos * 100

print("=" * 50)
print("COBERTURA DE LOS 30 AEROPUERTOS PRINCIPALES")
print("=" * 50)

print("Aeropuertos seleccionados:", top30_coordenadas.count())
print("Total de vuelos:", f"{total_vuelos:,}")
print("Vuelos de los 30 aeropuertos:", f"{vuelos_top30:,}")
print("Cobertura de vuelos:", f"{cobertura:.2f}%")

# Verificamos que todos tengan coordenadas.
sin_coordenadas = top30_coordenadas.filter(
    F.col("latitude").isNull() |
    F.col("longitude").isNull()
).count()

print("Aeropuertos sin coordenadas:", sin_coordenadas)

display(top30_coordenadas)


In [0]:

# ============================================================
# CELDA 15. GUARDAR LOS 30 AEROPUERTOS SELECCIONADOS
# ============================================================

# Guardamos la selección definitiva de aeropuertos
# para utilizarla en las descargas meteorológicas.
#
# Esto permite recuperar la lista aunque reinicie Python.
# No modificamos las tablas originales de vuelos.

from pyspark.sql import functions as F

TABLA_TOP30 = "workspace.bronze_flights.weather_airports_top30"

# Recuperamos el resultado de la celda anterior.
df_top30 = top30_coordenadas.select(
    "airport",
    "total_vuelos",
    "latitude",
    "longitude",
    "timezone"
)

# Validamos el número de aeropuertos.
if df_top30.count() != 30:
    raise ValueError("La selección no contiene 30 aeropuertos.")

# Evitamos crear una tabla duplicada.
if not spark.catalog.tableExists(TABLA_TOP30):

    (
        df_top30.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_TOP30)
    )

    print("Tabla creada correctamente.")

else:
    print("La tabla ya existe. No se sobrescribió.")

# Comprobamos la selección guardada.
df_guardado = spark.table(TABLA_TOP30)

total = df_guardado.count()
unicos = df_guardado.select("airport").distinct().count()

print("Aeropuertos guardados:", total)
print("Aeropuertos únicos:", unicos)

if total == 30 and unicos == 30:
    print("VALIDACIÓN EXITOSA: selección guardada.")
else:
    print("REVISAR: la tabla tiene registros inesperados.")

display(df_guardado.orderBy(F.desc("total_vuelos")))


In [0]:

# ============================================================
# CELDA 16. PRIMER LOTE DEFINITIVO DE CLIMA
# ============================================================

# Descargamos febrero de 2024 para 10 aeropuertos
# del Top 30 de BTS.
#
# Guardamos los datos en Delta usando MERGE.
# Si repetimos la celda, no insertará nuevamente
# los mismos aeropuertos y horas.

import requests
import pandas as pd

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

TABLA_TOP30 = "workspace.bronze_flights.weather_airports_top30"
TABLA_CLIMA = "workspace.bronze_flights.weather_forecasts_raw"

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

ANIO = 2024
MES = 2
FECHA_INICIO = "2024-02-01"
FECHA_FIN = "2024-02-29"

VARIABLES = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

# Recuperamos los 10 aeropuertos con mayor volumen
# dentro del Top 30 ya guardado en Databricks.
lote = (
    spark.table(TABLA_TOP30)
    .orderBy(F.desc("total_vuelos"), F.asc("airport"))
    .limit(10)
    .toPandas()
)

print("Aeropuertos del lote:", lote["airport"].tolist())

# ------------------------------------------------------------
# 2. CONSULTAR OPEN-METEO
# ------------------------------------------------------------

parametros = {
    "latitude": ",".join(lote["latitude"].astype(str)),
    "longitude": ",".join(lote["longitude"].astype(str)),
    "start_date": FECHA_INICIO,
    "end_date": FECHA_FIN,
    "hourly": ",".join(
        f"{v}_previous_day1" for v in VARIABLES
    ),
    "timezone": "UTC",
    "models": "gfs_global"
}

respuesta = requests.get(
    URL,
    params=parametros,
    timeout=180
)

respuesta.raise_for_status()
datos = respuesta.json()

if not isinstance(datos, list) or len(datos) != 10:
    raise ValueError(
        "La API no devolvió los 10 aeropuertos esperados."
    )

# ------------------------------------------------------------
# 3. PREPARAR Y VALIDAR LOS REGISTROS
# ------------------------------------------------------------

registros = []

# Febrero de 2024 tiene 29 días.
HORAS_ESPERADAS = 29 * 24

for indice, ubicacion in enumerate(datos):

    aeropuerto = lote.iloc[indice]["airport"]
    horarios = ubicacion.get("hourly", {})
    fechas = horarios.get("time", [])

    if len(fechas) != HORAS_ESPERADAS:
        raise ValueError(
            f"{aeropuerto}: cantidad de horas incorrecta."
        )

    for variable in VARIABLES:
        columna = f"{variable}_previous_day1"
        valores = horarios.get(columna, [])

        if (
            len(valores) != HORAS_ESPERADAS
            or any(v is None for v in valores)
        ):
            raise ValueError(
                f"{aeropuerto}: datos incompletos en {columna}."
            )

    for posicion, fecha in enumerate(fechas):

        fila = {
            "airport": aeropuerto,
            "latitude": float(lote.iloc[indice]["latitude"]),
            "longitude": float(lote.iloc[indice]["longitude"]),
            "forecast_valid_time_utc": fecha,
            "source_model": "gfs_global",
            "source_api": "open_meteo_previous_runs",
            "forecast_lead_hours": 24,
            "source_year": ANIO,
            "source_month": MES
        }

        for variable in VARIABLES:
            columna = f"{variable}_previous_day1"
            fila[columna] = float(
                horarios[columna][posicion]
            )

        registros.append(fila)

df_pandas = pd.DataFrame(registros)

# Comprobamos que existan 696 horas por aeropuerto.
if len(df_pandas) != 6960:
    raise ValueError(
        f"Se esperaban 6960 registros y llegaron "
        f"{len(df_pandas)}."
    )

# Comprobamos que no haya aeropuertos y horas repetidos.
if df_pandas.duplicated(
    ["airport", "forecast_valid_time_utc"]
).any():
    raise ValueError("El lote contiene registros duplicados.")

print("Registros descargados:", len(df_pandas))

# ------------------------------------------------------------
# 4. CONVERTIR A SPARK
# ------------------------------------------------------------

df_spark = (
    spark.createDataFrame(df_pandas)
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

# Ajustamos los datos al esquema de nuestra tabla Bronze.
esquema_destino = spark.table(TABLA_CLIMA).schema

df_spark = df_spark.select([
    F.col(campo.name)
    .cast(campo.dataType)
    .alias(campo.name)
    for campo in esquema_destino.fields
])

# ------------------------------------------------------------
# 5. INSERTAR ÚNICAMENTE REGISTROS NUEVOS
# ------------------------------------------------------------

df_spark.createOrReplaceTempView(
    "weather_feb2024_batch"
)

columnas = df_spark.columns

columnas_sql = ", ".join(
    f"`{c}`" for c in columnas
)

valores_sql = ", ".join(
    f"s.`{c}`" for c in columnas
)

spark.sql(f"""
MERGE INTO {TABLA_CLIMA} AS t
USING weather_feb2024_batch AS s

ON t.airport = s.airport
AND t.forecast_valid_time_utc = s.forecast_valid_time_utc
AND t.source_model = s.source_model
AND t.forecast_lead_hours = s.forecast_lead_hours

WHEN NOT MATCHED THEN
INSERT ({columnas_sql})
VALUES ({valores_sql})
""")

# ------------------------------------------------------------
# 6. VERIFICAR EL LOTE GUARDADO
# ------------------------------------------------------------

codigos_lote = lote["airport"].tolist()

df_guardado = (
    spark.table(TABLA_CLIMA)
    .filter(F.col("source_year") == ANIO)
    .filter(F.col("source_month") == MES)
    .filter(F.col("airport").isin(codigos_lote))
    .filter(F.col("source_model") == "gfs_global")
    .filter(F.col("forecast_lead_hours") == 24)
)

total_guardado = df_guardado.count()

print("\nRESULTADO DEL LOTE")
print("-" * 45)
print("Aeropuertos:", len(codigos_lote))
print("Mes:", FECHA_INICIO[:7])
print("Registros guardados:", total_guardado)
print("Registros esperados:", 6960)

display(
    df_guardado
    .groupBy("airport")
    .count()
    .orderBy("airport")
)

if total_guardado == 6960:
    print("VALIDACIÓN EXITOSA: lote guardado en Bronze.")
else:
    print("REVISAR: la cantidad guardada no coincide.")


In [0]:

# ============================================================
# CELDA 17. INGESTA COMPLETA DE PRONÓSTICOS A BRONZE
# ============================================================

# Objetivo:
# Descargar pronósticos horarios para los 30 aeropuertos
# seleccionados, de febrero de 2024 a diciembre de 2025.
#
# El proceso:
# 1. Recupera los aeropuertos desde Delta Lake.
# 2. Trabaja por mes y lotes de hasta 10 aeropuertos.
# 3. Omite aeropuertos/meses que ya estén completos.
# 4. Valida las respuestas de Open-Meteo.
# 5. Inserta registros nuevos mediante MERGE.
# 6. Se detiene ante errores para poder reanudarse.
#
# No modifica la tabla original de vuelos BTS.

import requests
import pandas as pd
import calendar
import time

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

TABLA_CLIMA = "workspace.bronze_flights.weather_forecasts_raw"
TABLA_TOP30 = "workspace.bronze_flights.weather_airports_top30"

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

VARIABLES = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

MODELO = "gfs_global"
HORIZONTE = 24
TAMANO_LOTE = 10

# Pausa entre solicitudes para reducir la carga de API.
PAUSA_SEGUNDOS = 5

# Todos los meses entre febrero de 2024 y diciembre de 2025.
PERIODOS = [
    (periodo.year, periodo.month)
    for periodo in pd.period_range(
        "2024-02", "2025-12", freq="M"
    )
]

# Recuperamos los 30 aeropuertos guardados.
aeropuertos = (
    spark.table(TABLA_TOP30)
    .orderBy(F.desc("total_vuelos"), F.asc("airport"))
    .toPandas()
)

if len(aeropuertos) != 30:
    raise ValueError("La tabla no contiene 30 aeropuertos.")

if aeropuertos["airport"].duplicated().any():
    raise ValueError("Hay códigos de aeropuerto repetidos.")

if aeropuertos[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Hay aeropuertos sin coordenadas.")

codigos_validos = aeropuertos["airport"].tolist()

print("Aeropuertos:", len(codigos_validos))
print("Meses:", len(PERIODOS))
print("Modelo:", MODELO)
print("Horizonte:", HORIZONTE, "horas")

# ------------------------------------------------------------
# 2. RECUPERAR EL PROGRESO EXISTENTE
# ------------------------------------------------------------

# Contamos las horas ya guardadas por aeropuerto y mes.
# Consideramos exclusivamente nuestro modelo y horizonte.

conteos = (
    spark.table(TABLA_CLIMA)
    .filter(F.col("airport").isin(codigos_validos))
    .filter(F.col("source_model") == MODELO)
    .filter(F.col("forecast_lead_hours") == HORIZONTE)
    .groupBy("airport", "source_year", "source_month")
    .agg(
        F.countDistinct("forecast_valid_time_utc")
        .alias("horas_guardadas")
    )
    .collect()
)

progreso = {
    (
        fila["airport"],
        fila["source_year"],
        fila["source_month"]
    ): fila["horas_guardadas"]
    for fila in conteos
}

print("Periodos de aeropuerto con registros previos:",
      len(progreso))

# ------------------------------------------------------------
# 3. FUNCIÓN PARA CONSULTAR OPEN-METEO
# ------------------------------------------------------------

def consultar_open_meteo(lote, inicio, fin):

    parametros = {
        "latitude": ",".join(
            lote["latitude"].astype(str)
        ),
        "longitude": ",".join(
            lote["longitude"].astype(str)
        ),
        "start_date": inicio,
        "end_date": fin,
        "hourly": ",".join(
            f"{v}_previous_day1" for v in VARIABLES
        ),
        "timezone": "UTC",
        "models": MODELO
    }

    # Reintentamos errores temporales.
    # Si la API devuelve un límite de solicitudes,
    # detenemos la ingesta para no insistir.
    for intento in range(1, 4):

        try:
            respuesta = requests.get(
                URL,
                params=parametros,
                timeout=180
            )

            if respuesta.status_code == 429:
                raise RuntimeError(
                    "Open-Meteo limitó las solicitudes (HTTP 429). "
                    "Detener la carga y reanudar posteriormente."
                )

            respuesta.raise_for_status()
            datos = respuesta.json()

            if isinstance(datos, dict) and datos.get("error"):
                raise RuntimeError(
                    datos.get("reason", "Error de Open-Meteo")
                )

            if not isinstance(datos, list):
                raise ValueError(
                    "Se esperaba una lista de ubicaciones."
                )

            if len(datos) != len(lote):
                raise ValueError(
                    "La cantidad de ubicaciones no coincide."
                )

            return datos

        except requests.RequestException as error:

            if intento == 3:
                raise

            print(
                f"Error de conexión. Reintento {intento}/3:",
                error
            )
            time.sleep(15 * intento)

# ------------------------------------------------------------
# 4. FUNCIÓN PARA GUARDAR UN LOTE
# ------------------------------------------------------------

def guardar_lote(lote, datos, anio, mes, horas_esperadas):

    registros = []

    for indice, ubicacion in enumerate(datos):

        aeropuerto = str(lote.iloc[indice]["airport"])
        latitud = float(lote.iloc[indice]["latitude"])
        longitud = float(lote.iloc[indice]["longitude"])

        horarios = ubicacion.get("hourly", {})
        fechas = horarios.get("time", [])

        if len(fechas) != horas_esperadas:
            raise ValueError(
                f"{aeropuerto}: número de horas incorrecto."
            )

        if len(set(fechas)) != horas_esperadas:
            raise ValueError(
                f"{aeropuerto}: horas repetidas."
            )

        for variable in VARIABLES:

            columna = f"{variable}_previous_day1"
            valores = horarios.get(columna, [])

            if len(valores) != horas_esperadas:
                raise ValueError(
                    f"{aeropuerto}: longitud incorrecta "
                    f"en {columna}."
                )

            if any(v is None for v in valores):
                raise ValueError(
                    f"{aeropuerto}: faltan valores "
                    f"en {columna}."
                )

        for posicion, fecha in enumerate(fechas):

            fila = {
                "airport": aeropuerto,
                "latitude": latitud,
                "longitude": longitud,
                "forecast_valid_time_utc": fecha,
                "source_model": MODELO,
                "source_api": "open_meteo_previous_runs",
                "forecast_lead_hours": HORIZONTE,
                "source_year": anio,
                "source_month": mes
            }

            for variable in VARIABLES:

                columna = f"{variable}_previous_day1"
                fila[columna] = float(
                    horarios[columna][posicion]
                )

            registros.append(fila)

    df_pandas = pd.DataFrame(registros)

    esperados_lote = len(lote) * horas_esperadas

    if len(df_pandas) != esperados_lote:
        raise ValueError(
            "La cantidad total del lote no coincide."
        )

    llave = ["airport", "forecast_valid_time_utc"]

    if df_pandas.duplicated(llave).any():
        raise ValueError(
            "Se encontraron registros repetidos en el lote."
        )

    # Convertimos únicamente este lote a Spark.
    df_spark = (
        spark.createDataFrame(df_pandas)
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
    )

    # Mantenemos el esquema de la tabla existente.
    esquema = spark.table(TABLA_CLIMA).schema

    df_spark = df_spark.select([
        F.col(campo.name)
        .cast(campo.dataType)
        .alias(campo.name)
        for campo in esquema.fields
    ])

    df_spark.createOrReplaceTempView(
        "weather_ingestion_batch"
    )

    columnas_sql = ", ".join(
        f"`{c}`" for c in df_spark.columns
    )

    valores_sql = ", ".join(
        f"s.`{c}`" for c in df_spark.columns
    )

    # Insertamos solo las combinaciones que no existen.
    spark.sql(f"""
    MERGE INTO {TABLA_CLIMA} AS t
    USING weather_ingestion_batch AS s

    ON t.airport = s.airport
    AND t.forecast_valid_time_utc = s.forecast_valid_time_utc
    AND t.source_model = s.source_model
    AND t.forecast_lead_hours = s.forecast_lead_hours

    WHEN NOT MATCHED THEN
    INSERT ({columnas_sql})
    VALUES ({valores_sql})
    """)

    # Verificamos el total almacenado para este lote.
    codigos = lote["airport"].tolist()

    df_verificacion = (
        spark.table(TABLA_CLIMA)
        .filter(F.col("airport").isin(codigos))
        .filter(F.col("source_year") == anio)
        .filter(F.col("source_month") == mes)
        .filter(F.col("source_model") == MODELO)
        .filter(F.col("forecast_lead_hours") == HORIZONTE)
    )

    conteo = (
        df_verificacion
        .groupBy("airport")
        .agg(
            F.countDistinct("forecast_valid_time_utc")
            .alias("horas")
        )
        .collect()
    )

    horas_por_aeropuerto = {
        fila["airport"]: fila["horas"]
        for fila in conteo
    }

    for codigo in codigos:

        if horas_por_aeropuerto.get(codigo, 0) != horas_esperadas:
            raise ValueError(
                f"{codigo}: no se completaron las "
                f"{horas_esperadas} horas en Delta."
            )

    # Actualizamos el progreso solo después de validar.
    for codigo in codigos:
        progreso[(codigo, anio, mes)] = horas_esperadas

    return len(df_pandas)

# ------------------------------------------------------------
# 5. EJECUTAR INGESTA INCREMENTAL
# ------------------------------------------------------------

lotes_completados = 0
lotes_omitidos = 0
errores = []

print("\nINICIANDO INGESTA COMPLETA")
print("=" * 55)

detener = False

for anio, mes in PERIODOS:

    if detener:
        break

    ultimo_dia = calendar.monthrange(anio, mes)[1]
    horas_esperadas = ultimo_dia * 24

    inicio = f"{anio}-{mes:02d}-01"
    fin = f"{anio}-{mes:02d}-{ultimo_dia:02d}"

    # Revisamos todos los aeropuertos en grupos de 10.
    for posicion in range(0, len(aeropuertos), TAMANO_LOTE):

        grupo = aeropuertos.iloc[
            posicion:posicion + TAMANO_LOTE
        ].copy()

        # Quitamos del lote los aeropuertos que ya
        # tienen todas las horas del mes almacenadas.
        pendientes = grupo[
            grupo["airport"].apply(
                lambda codigo:
                    progreso.get((codigo, anio, mes), 0)
                    < horas_esperadas
            )
        ].reset_index(drop=True)

        if pendientes.empty:

            lotes_omitidos += 1

            print(
                f"{anio}-{mes:02d} | "
                f"Lote {posicion // TAMANO_LOTE + 1}: "
                "ya completo, se omite."
            )

            continue

        print(
            f"\nProcesando {anio}-{mes:02d} | "
            f"Lote {posicion // TAMANO_LOTE + 1} | "
            f"Aeropuertos pendientes: {len(pendientes)}"
        )

        try:

            # Consultamos los pronósticos del periodo.
            datos = consultar_open_meteo(
                pendientes, inicio, fin
            )

            # Guardamos y verificamos el lote.
            descargados = guardar_lote(
                pendientes,
                datos,
                anio,
                mes,
                horas_esperadas
            )

            lotes_completados += 1

            print(
                f"Lote guardado y validado: "
                f"{descargados:,} registros recibidos."
            )

        except Exception as error:

            print("ERROR:", str(error))

            errores.append({
                "anio": anio,
                "mes": mes,
                "lote": posicion // TAMANO_LOTE + 1,
                "error": str(error)
            })

            # Detenemos la ejecución. Los lotes que
            # ya terminaron permanecen guardados.
            detener = True
            break

        time.sleep(PAUSA_SEGUNDOS)

# ------------------------------------------------------------
# 6. RESUMEN
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RESUMEN DE INGESTA METEOROLÓGICA")
print("=" * 55)

print("Lotes completados:", lotes_completados)
print("Lotes omitidos por estar completos:", lotes_omitidos)
print("Errores encontrados:", len(errores))

if errores:
    display(pd.DataFrame(errores))
    print("Carga interrumpida. Los datos guardados se conservan.")
else:
    print("Recorrido de ingesta finalizado sin errores.")

# Mostramos registros presentes en el periodo del proyecto.
df_final = (
    spark.table(TABLA_CLIMA)
    .filter(
        (F.col("source_year") == 2025) |
        (
            (F.col("source_year") == 2024) &
            (F.col("source_month") >= 2)
        )
    )
    .filter(F.col("airport").isin(codigos_validos))
    .filter(F.col("source_model") == MODELO)
    .filter(F.col("forecast_lead_hours") == HORIZONTE)
)

print(
    "Registros en Bronze para el Top 30:",
    f"{df_final.count():,}"
)

print("Registros esperados al completar todo: 504,000")


In [0]:

# ============================================================
# CELDA 18. VALIDACIÓN FINAL DE BRONZE METEOROLÓGICO
# ============================================================

# Comprobamos que los 30 aeropuertos tengan todas
# las horas de los 23 meses y que no existan registros
# duplicados por aeropuerto, hora, modelo y horizonte.

import calendar
import pandas as pd
from pyspark.sql import functions as F

TABLA_CLIMA = "workspace.bronze_flights.weather_forecasts_raw"
TABLA_TOP30 = "workspace.bronze_flights.weather_airports_top30"

codigos = [
    fila["airport"]
    for fila in spark.table(TABLA_TOP30)
    .select("airport")
    .collect()
]

# Consultamos exclusivamente el modelo y el periodo
# utilizados en nuestro proyecto.
df = (
    spark.table(TABLA_CLIMA)
    .filter(F.col("airport").isin(codigos))
    .filter(F.col("source_model") == "gfs_global")
    .filter(F.col("forecast_lead_hours") == 24)
    .filter(
        (F.col("source_year") == 2025) |
        (
            (F.col("source_year") == 2024) &
            (F.col("source_month") >= 2)
        )
    )
)

# Calculamos cuántos registros y horas únicas hay
# por aeropuerto, año y mes.
resumen = (
    df.groupBy("airport", "source_year", "source_month")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("forecast_valid_time_utc")
        .alias("horas_unicas")
    )
)

# Construimos la lista de periodos esperados.
periodos = [
    (p.year, p.month)
    for p in pd.period_range("2024-02", "2025-12", freq="M")
]

esperados = {
    (codigo, anio, mes):
    calendar.monthrange(anio, mes)[1] * 24
    for codigo in codigos
    for anio, mes in periodos
}

observados = {
    (
        fila["airport"],
        fila["source_year"],
        fila["source_month"]
    ): (fila["registros"], fila["horas_unicas"])
    for fila in resumen.collect()
}

faltantes = [
    clave for clave in esperados
    if clave not in observados
]

incompletos = [
    clave for clave, horas in esperados.items()
    if clave in observados
    and observados[clave][1] != horas
]

duplicados = [
    clave for clave, valores in observados.items()
    if valores[0] != valores[1]
]

print("=" * 50)
print("VALIDACIÓN FINAL BRONZE METEOROLÓGICO")
print("=" * 50)

print("Aeropuertos esperados:", len(codigos))
print("Periodos esperados:", len(esperados))
print("Periodos encontrados:", len(observados))
print("Periodos faltantes:", len(faltantes))
print("Periodos incompletos:", len(incompletos))
print("Periodos con duplicados:", len(duplicados))
print("Registros totales:", f"{df.count():,}")

if (
    len(faltantes) == 0
    and len(incompletos) == 0
    and len(duplicados) == 0
    and len(observados) == 690
):
    print("VALIDACIÓN EXITOSA: Bronze meteorológico completo.")
else:
    print("REVISAR: se encontraron inconsistencias.")
    print("Faltantes:", faltantes[:10])
    print("Incompletos:", incompletos[:10])
    print("Duplicados:", duplicados[:10])
